In [ ]:
import pandas as pd
import numpy as np
import keras
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from keras import layers
from keras.utils import to_categorical
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
first_df = 

In [ ]:
def parse_data_hex(hexstr, dlc):
    # many CSVs provide e.g. '01FFA0' or '01 FF A0', handle both
    if pd.isna(hexstr):
        return [0]*8
    s = str(hexstr).replace(" ", "")
    # if csv stores bytes separated by commas, try to handle
    s = s.replace(",", "")
    # ensure even length
    if len(s) % 2 != 0:
        s = s + "0"
    bytes_list = []
    try:
        for i in range(0, min(16, len(s)), 2):
            b = int(s[i:i+2], 16)
            bytes_list.append(b)
    except Exception:
        # fallback: zeros
        bytes_list = []
    # pad to 8
    while len(bytes_list) < 8:
        bytes_list.append(0)
    return bytes_list[:8]


In [ ]:
# load a single CSV into standardized dataframe columns:
def load_signal_csv(path):
    # expected columns: timestamp (s or ms), id, dlc, data (hex)
    df = pd.read_csv(path)
    # Normalize column names
    cols = {c.lower(): c for c in df.columns}
    # find timestamp col
    tcol = None
    for cand in ["timestamp", "time", "ts", "relative_time", "relative timestamp"]:
        if cand in cols:
            tcol = cols[cand]; break
    if tcol is None:
        raise ValueError(f"no timestamp col in {path}. cols={df.columns}")
    # find id, dlc, data columns heuristically
    idcol = cols.get("id", None) or cols.get("arbitration_id", None) or cols.get("can_id", None)
    dlccol = cols.get("dlc", None)
    datacol = cols.get("data", None) or cols.get("payload", None) or cols.get("data_hex", None)
    if idcol is None or datacol is None:
        raise ValueError(f"couldn't find id/data cols for {path}")
    # standardize
    df2 = pd.DataFrame()
    df2["timestamp"] = df[tcol].astype(float)
    df2["id"] = df[idcol]
    df2["dlc"] = df[dlccol] if dlccol in df.columns else df[datacol].apply(lambda x: len(str(x).replace(" ", ""))//2)
    df2["data_raw"] = df[datacol].astype(str)
    # ensure timestamps are relative seconds (some files may be ms)
    # Heuristic: if max timestamp > 1e5 assume ms -> convert to s
    if df2["timestamp"].max() > 1e5:
        df2["timestamp"] = df2["timestamp"] / 1000.0
    # delta time
    df2 = df2.sort_values("timestamp").reset_index(drop=True)
    df2["delta_t"] = df2["timestamp"].diff().fillna(0.0)
    # parse data bytes
    data_bytes = df2["data_raw"].apply(lambda h: parse_data_hex(h, 8))
    data_cols = pd.DataFrame(data_bytes.tolist(), columns=[f"b{i}" for i in range(8)])
    df3 = pd.concat([df2[["timestamp","id","dlc","delta_t"]].reset_index(drop=True), data_cols], axis=1)
    # ensure id numeric (strip 0x)
    df3["id"] = df3["id"].astype(str).str.replace("0x","").apply(lambda x: int(x,16) if all(c in "0123456789abcdefABCDEF" for c in x) else int(x))
    return df3


In [ ]:

# label frames given metadata (file_key refers to the filename base without extension)
def label_frames(df_frames, file_key, is_attack):
    # default label = 0
    df = df_frames.copy()
    df["label"] = 0
    if not is_attack:
        return df
    # find attack metadata entry
    meta = attack_meta.get(file_key, None)
    if meta is None:
        # if not found, label whole file as attack (conservative)
        df["label"] = 1
        return df
    inv = meta.get("injection_interval", None)
    if inv is None:
        # no injection interval given -> label entire file attack (or change policy)
        df["label"] = 1
        return df
    # injection_interval is [start_sec, end_sec]
    start, end = float(inv[0]), float(inv[1])
    # timestamps assumed relative to start of file (0 = capture start)
    df.loc[(df["timestamp"] >= start) & (df["timestamp"] <= end), "label"] = 1
    return df


In [ ]:


# convert single file dataframe into windows
def frames_to_windows(df, window_len=WINDOW_LEN, stride=STRIDE):
    feats = []
    labels = []
    N = len(df)
    for start in range(0, max(1,N-window_len+1), stride):
        win = df.iloc[start:start+window_len]
        if len(win) < window_len:
            continue
        # create feature array: [window_len, num_features]
        # features per frame: id_norm, dlc_norm, b0..b7_norm, is_extended(0), delta_t_norm
        id_norm = win["id"].values / float(MAX_ID)
        dlc = win["dlc"].values.astype(float) / 8.0
        bytes_arr = win[[f"b{i}" for i in range(8)]].values / 255.0
        is_ext = np.zeros((window_len,1))  # placeholder; if you have extended flags, use them
        dt = win["delta_t"].values.reshape(-1,1)
        # scale dt (clip to e.g. 1s)
        dt = np.clip(dt, 0.0, 1.0)
        dt = dt.reshape(-1,1)
        frame_feats = np.hstack([id_norm.reshape(-1,1), dlc.reshape(-1,1), bytes_arr, is_ext, dt])
        feats.append(frame_feats)
        labels.append(1 if win["label"].any() else 0)
    if len(feats) == 0:
        return np.empty((0,window_len,12)), np.array([], dtype=int)
    return np.stack(feats), np.array(labels, dtype=int)

In [ ]:
# ---------- MAIN: load all files, label, window ----------
def gather_all_windows():
    X_list = []
    y_list = []
    file_list = []  # to allow file-level split later
    # ambient
    csvs = glob(os.path.join(AMBIENT_DIR, "*.csv"))
    for p in tqdm(csvs, desc="ambient csvs"):
        fname = os.path.splitext(os.path.basename(p))[0]  # matches ambient metadata keys
        df = load_signal_csv(p)
        dfL = label_frames(df, fname, is_attack=False)
        Xw, yw = frames_to_windows(dfL)
        if Xw.shape[0] > 0:
            X_list.append(Xw); y_list.append(yw); file_list += [fname]*len(yw)
    # attacks
    csvs = glob(os.path.join(ATTACKS_DIR, "*.csv"))
    for p in tqdm(csvs, desc="attack csvs"):
        fname = os.path.splitext(os.path.basename(p))[0]
        df = load_signal_csv(p)
        dfL = label_frames(df, fname, is_attack=True)
        Xw, yw = frames_to_windows(dfL)
        if Xw.shape[0] > 0:
            X_list.append(Xw); y_list.append(yw); file_list += [fname]*len(yw)
    if not X_list:
        return None, None, None
    X = np.concatenate(X_list, axis=0)  # shape (samples, window_len, features)
    y = np.concatenate(y_list, axis=0)
    return X, y, file_list


In [ ]:

# run gather
X, y, files = gather_all_windows()
print("X shape:", None if X is None else X.shape, "y counts:", None if y is None else np.bincount(y))

# Save arrays for later training
os.makedirs("processed", exist_ok=True)
np.save("processed/X_windows.npy", X)
np.save("processed/y_windows.npy", y)
with open("processed/files_list.json","w") as f:
    json.dump(files, f)

# ---------- Train/test split by file (prevent leakage) ----------
# load files_list to map windows -> file. We used same length allocation earlier.
files_arr = np.array(files)
unique_files = np.unique(files_arr)
train_files, test_files = train_test_split(unique_files, test_size=0.2, random_state=42)
# map windows to file index
is_train = np.isin(files_arr, train_files)
X_train, X_test = X[is_train], X[~is_train]
y_train, y_test = y[is_train], y[~is_train]
print("train/test shapes:", X_train.shape, X_test.shape)

# balance: compute class weights
from sklearn.utils import class_weight
class_weights = class_weight.compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights_dict = {i: w for i,w in enumerate(class_weights)}
print("class weights:", class_weights_dict)

# reshape for CNN: add channel axis
X_train = X_train[..., np.newaxis]
X_test  = X_test[..., np.newaxis]


In [ ]:
def build_tiny_cnn(input_shape):
    model = models.Sequential()
    # one conv layer -> global pooling -> dense
    model.add(layers.Conv2D(16, kernel_size=(5, input_shape[1]), activation='relu', input_shape=input_shape))
    # The Conv2D uses kernel height along time dimension and width across features to capture patterns across features
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(32, activation='relu'))
    model.add(layers.Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
# input shape: (window_len, num_features, 1)
input_shape = X_train.shape[1:]
model = build_tiny_cnn(input_shape)
model.summary()

# training (small example)
history = model.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.2, class_weight=class_weights_dict)

# evaluate
print("Test eval:", model.evaluate(X_test, y_test))

In [ ]:
#optional: 
import cantools
db = cantools.database.load_file("signal_extractions/DBC/your_file.dbc")
# decode a frame:
frame_id = 0x123
data_bytes = bytes([0x01,0x02, ...])
msg = db.get_message_by_frame_id(frame_id)
signals = msg.decode(data_bytes)  # dict of signal_name -> value